# Pratica 01 — Dal notebook alla produzione

**Corso:** MLOps e Monitoraggio con Python · **Lezione 01** · **Tempo stimato:** 40-55 minuti
**Notebook di lezione:** `notebooks_clase/clase01_del_notebook_a_produccion_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio parte da un frammento di codice con **lo stesso tipo di problema** che hai visto in `datos/notebook_sucio.ipynb`. Il tuo lavoro è correggerlo, non addestrare un modello vero: i dati sono piccoli array giocattolo.
- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas` e `scikit-learn` (solo `DummyClassifier` e `train_test_split`).

---
## Cella 0 — Setup

In [ ]:
import os
import inspect

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

print("Librerie caricate correttamente.")

## Esercizio 1 — Togliere il percorso assoluto hardcoded

`ruta_datos_clientes` è copiata da un notebook reale: restituisce sempre lo stesso percorso assoluto del portatile di chi l'ha scritta. Completala perché:

- riceva `nombre` come parametro (con `"clientes.csv"` di default),
- restituisca `os.path.join("datos", nombre)` invece di un percorso assoluto fisso.

In [ ]:
def ruta_datos_clientes():
    return "/Users/alguien/proyectos/churn/datos/clientes.csv"      # TODO: parametro «nombre» + os.path.join("datos", nombre)

# Controllo automatico
firma = inspect.signature(ruta_datos_clientes)
assert "nombre" in firma.parameters, "Completa le righe contrassegnate con TODO"
r = ruta_datos_clientes()
assert r == os.path.join("datos", "clientes.csv"), "Completa le righe contrassegnate con TODO"
assert not r.startswith("/Users") and not r.startswith("/home") and ":\\" not in r, "il percorso non deve essere assoluto né dipendere da un utente specifico"
assert ruta_datos_clientes("otro.csv") == os.path.join("datos", "otro.csv"), "il parametro «nombre» deve essere usato davvero"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Aggiungere `random_state` a una divisione riutilizzabile

`dividir` viene chiamata da diversi punti del progetto, ma non fissa nessun seme: ogni chiamata ripartisce le righe in modo diverso. Completala perché riceva `semilla` e la passi a `train_test_split`.

In [ ]:
def dividir(X, y, semilla=42):
    return train_test_split(X, y, test_size=0.25)      # TODO: passa random_state=semilla

# Controllo automatico
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
a1, a2, a3, a4 = dividir(X_juguete, y_juguete, semilla=7)
b1, b2, b3, b4 = dividir(X_juguete, y_juguete, semilla=7)
assert np.array_equal(a1, b1) and np.array_equal(a2, b2), "Completa le righe contrassegnate con TODO"
c1, c2, c3, c4 = dividir(X_juguete, y_juguete, semilla=99)
assert not np.array_equal(a1, c1), "con semi diversi anche la ripartizione dovrebbe essere diversa"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Togliere la soglia da una variabile globale

`en_riesgo` legge `umbral_global`, una variabile definita fuori dalla funzione. Se qualcuno la sovrascrive più avanti (come succedeva con `umbral_riesgo` nel notebook sporco), il risultato di `en_riesgo` cambia senza che la funzione sia cambiata. Completala perché riceva `umbral` come parametro, con `0.5` di default, e smetta di dipendere dalla variabile globale.

In [ ]:
umbral_global = 0.5


def en_riesgo(prob):
    return prob > umbral_global      # TODO: parametro «umbral=0.5», senza usare umbral_global

# Controllo automatico
firma = inspect.signature(en_riesgo)
assert "umbral" in firma.parameters, "Completa le righe contrassegnate con TODO"
assert en_riesgo(0.6, umbral=0.5) is True and en_riesgo(0.4, umbral=0.5) is False, "Completa le righe contrassegnate con TODO"
umbral_global = 0.9      # qualcuno sovrascrive la globale più avanti, come nel notebook sporco
assert en_riesgo(0.6, umbral=0.5) is True, "la funzione non deve più dipendere da umbral_global"
assert en_riesgo(0.6) is True, "anche con il valore di default (0.5) deve dare True per 0.6"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Non buttare via il modello addestrato

`entrenar_y_tirar` addestra un modello e non lo restituisce: come nella cella finale del notebook sporco, il modello resta solo in memoria dentro la funzione e si perde non appena questa termina. Completala perché **restituisca il modello addestrato**.

In [ ]:
def entrenar_y_tirar(X, y):
    modelo = DummyClassifier(strategy="most_frequent").fit(X, y)
    print("modelo entrenado")      # TODO: restituisci «modelo» invece di perderlo

# Controllo automatico
X_juguete = np.arange(20).reshape(10, 2)
y_juguete = np.array([0, 0, 0, 1, 0, 0, 1, 0, 0, 0])
resultado = entrenar_y_tirar(X_juguete, y_juguete)
assert resultado is not None, "Completa le righe contrassegnate con TODO"
assert hasattr(resultado, "predict"), "deve restituire il modello, non None né una metrica"
assert list(resultado.predict(X_juguete)) == [0] * 10, "con strategy=\"most_frequent\" predice sempre la classe maggioritaria (0)"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Validare i dati prima di usarli

`cargar_validado` riceve un `DataFrame` e lo restituisce così com'è, senza controllare nulla — come il commento "non serve verificare nulli" del notebook sporco. Completala perché sollevi `ValueError` se:

- manca qualche colonna di `COLUMNAS_REQUERIDAS`, oppure
- c'è qualche valore nullo in `"antiguedad_meses"`.

Se tutto è a posto, deve restituire il `DataFrame` senza modificarlo.

In [ ]:
COLUMNAS_REQUERIDAS = ["antiguedad_meses", "uso_mensual_horas", "abandono"]


def cargar_validado(df):
    return df      # TODO: ValueError se mancano colonne o ci sono nulli in antiguedad_meses

# Controllo automatico
df_ok = pd.DataFrame({"antiguedad_meses": [1, 2, 3], "uso_mensual_horas": [10, 20, 30], "abandono": [0, 1, 0]})
assert cargar_validado(df_ok) is not None and len(cargar_validado(df_ok)) == 3, "Completa le righe contrassegnate con TODO"
df_sin_columna = df_ok.drop(columns=["uso_mensual_horas"])
try:
    cargar_validado(df_sin_columna)
    raise AssertionError("doveva sollevare ValueError se manca una colonna richiesta")
except ValueError:
    pass
df_con_nulo = df_ok.copy()
df_con_nulo.loc[0, "antiguedad_meses"] = None
try:
    cargar_validado(df_con_nulo)
    raise AssertionError("doveva sollevare ValueError se ci sono nulli in antiguedad_meses")
except ValueError:
    pass
print("Esercizio 5 corretto ✔")

---
## Sfida finale — un `pipeline_reproducible` senza nessuno dei problemi del notebook sporco

Integra quanto appreso. Completa `pipeline_reproducible(X, y, semilla)`, che deve:

- dividere con `dividir(X, y, semilla)` (esercizio 2),
- addestrare un `DummyClassifier(strategy="most_frequent")` con l'addestramento, **senza buttarlo via** (esercizio 4),
- restituire `(modelo, X_prueba, y_prueba)`.

Nessun percorso assoluto, nessuna variabile globale, nessun seme assente: tutto il necessario entra come parametro.

In [ ]:
def pipeline_reproducible(X, y, semilla):
    X_ent, X_prueba, y_ent, y_prueba = None, None, None, None      # TODO: usa dividir(X, y, semilla)
    modelo = None      # TODO: addestra un DummyClassifier(strategy="most_frequent") con X_ent, y_ent
    return modelo, X_prueba, y_prueba

# Controllo automatico
X_juguete = np.arange(60).reshape(30, 2)
y_juguete = np.array(([0] * 20) + ([1] * 10))
resultado = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert resultado is not None, "Completa le righe contrassegnate con TODO"
modelo, X_prueba, y_prueba = resultado
assert hasattr(modelo, "predict"), "restituisci il modello addestrato, non None"
modelo_b, X_prueba_b, y_prueba_b = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert np.array_equal(X_prueba, X_prueba_b), "con lo stesso seme la partizione di test deve ripetersi"
assert list(modelo.predict(X_prueba)) == list(modelo_b.predict(X_prueba_b)), "con lo stesso seme le predizioni devono coincidere"
print("Sfida finale corretta ✔")

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def ruta_datos_clientes(nombre="clientes.csv"):
    return os.path.join("datos", nombre)

# Controllo automatico
firma = inspect.signature(ruta_datos_clientes)
assert "nombre" in firma.parameters, "Completa le righe contrassegnate con TODO"
r = ruta_datos_clientes()
assert r == os.path.join("datos", "clientes.csv"), "Completa le righe contrassegnate con TODO"
assert not r.startswith("/Users") and not r.startswith("/home") and ":\\" not in r, "il percorso non deve essere assoluto né dipendere da un utente specifico"
assert ruta_datos_clientes("otro.csv") == os.path.join("datos", "otro.csv"), "il parametro «nombre» deve essere usato davvero"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def dividir(X, y, semilla=42):
    return train_test_split(X, y, test_size=0.25, random_state=semilla)

# Controllo automatico
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
a1, a2, a3, a4 = dividir(X_juguete, y_juguete, semilla=7)
b1, b2, b3, b4 = dividir(X_juguete, y_juguete, semilla=7)
assert np.array_equal(a1, b1) and np.array_equal(a2, b2), "Completa le righe contrassegnate con TODO"
c1, c2, c3, c4 = dividir(X_juguete, y_juguete, semilla=99)
assert not np.array_equal(a1, c1), "con semi diversi anche la ripartizione dovrebbe essere diversa"
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
umbral_global = 0.5


def en_riesgo(prob, umbral=0.5):
    return prob > umbral

# Controllo automatico
firma = inspect.signature(en_riesgo)
assert "umbral" in firma.parameters, "Completa le righe contrassegnate con TODO"
assert en_riesgo(0.6, umbral=0.5) is True and en_riesgo(0.4, umbral=0.5) is False, "Completa le righe contrassegnate con TODO"
umbral_global = 0.9      # qualcuno sovrascrive la globale più avanti, come nel notebook sporco
assert en_riesgo(0.6, umbral=0.5) is True, "la funzione non deve più dipendere da umbral_global"
assert en_riesgo(0.6) is True, "anche con il valore di default (0.5) deve dare True per 0.6"
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def entrenar_y_tirar(X, y):
    modelo = DummyClassifier(strategy="most_frequent").fit(X, y)
    print("modelo entrenado")
    return modelo

# Controllo automatico
X_juguete = np.arange(20).reshape(10, 2)
y_juguete = np.array([0, 0, 0, 1, 0, 0, 1, 0, 0, 0])
resultado = entrenar_y_tirar(X_juguete, y_juguete)
assert resultado is not None, "Completa le righe contrassegnate con TODO"
assert hasattr(resultado, "predict"), "deve restituire il modello, non None né una metrica"
assert list(resultado.predict(X_juguete)) == [0] * 10, "con strategy=\"most_frequent\" predice sempre la classe maggioritaria (0)"
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
COLUMNAS_REQUERIDAS = ["antiguedad_meses", "uso_mensual_horas", "abandono"]


def cargar_validado(df):
    faltantes = [c for c in COLUMNAS_REQUERIDAS if c not in df.columns]
    if faltantes:
        raise ValueError("mancano colonne: " + str(faltantes))
    if df["antiguedad_meses"].isnull().any():
        raise ValueError("ci sono valori nulli in antiguedad_meses")
    return df

# Controllo automatico
df_ok = pd.DataFrame({"antiguedad_meses": [1, 2, 3], "uso_mensual_horas": [10, 20, 30], "abandono": [0, 1, 0]})
assert cargar_validado(df_ok) is not None and len(cargar_validado(df_ok)) == 3, "Completa le righe contrassegnate con TODO"
df_sin_columna = df_ok.drop(columns=["uso_mensual_horas"])
try:
    cargar_validado(df_sin_columna)
    raise AssertionError("doveva sollevare ValueError se manca una colonna richiesta")
except ValueError:
    pass
df_con_nulo = df_ok.copy()
df_con_nulo.loc[0, "antiguedad_meses"] = None
try:
    cargar_validado(df_con_nulo)
    raise AssertionError("doveva sollevare ValueError se ci sono nulli in antiguedad_meses")
except ValueError:
    pass
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def pipeline_reproducible(X, y, semilla):
    X_ent, X_prueba, y_ent, y_prueba = dividir(X, y, semilla)
    modelo = entrenar_y_tirar(X_ent, y_ent)
    return modelo, X_prueba, y_prueba

# Controllo automatico
X_juguete = np.arange(60).reshape(30, 2)
y_juguete = np.array(([0] * 20) + ([1] * 10))
resultado = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert resultado is not None, "Completa le righe contrassegnate con TODO"
modelo, X_prueba, y_prueba = resultado
assert hasattr(modelo, "predict"), "restituisci il modello addestrato, non None"
modelo_b, X_prueba_b, y_prueba_b = pipeline_reproducible(X_juguete, y_juguete, semilla=13)
assert np.array_equal(X_prueba, X_prueba_b), "con lo stesso seme la partizione di test deve ripetersi"
assert list(modelo.predict(X_prueba)) == list(modelo_b.predict(X_prueba_b)), "con lo stesso seme le predizioni devono coincidere"
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (debito tecnico specifico del ML, livelli di maturità, ordine di esecuzione delle celle, variabili globali, validazione dei dati)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*